# ASTRA-GUARD: Phase 4 — Disaster Intelligence AI Exploration
**AI-Powered Integrated Space Mission Safety & Intelligence**
Team OMEN · AITHON 2.0 · Track 11 Aerospace, Telemetry & SpaceTech

This notebook demonstrates:
1. **Sentinel-1 SAR / Sen1Floods11 Flood Segmentation:** Lightweight 2-level U-Net inference on synthetic and radar-like surface backscatter.
2. **Multi-Temporal SAR Change Detection:** Pre- vs. post-flood backscatter drop analysis quantifying inundation expansion and surface area.
3. **Dynamic Observation Priority Scoring ($r_e$):** Multi-factor priority weighting (surface area, recency, population proxy, cloud clearance).
4. **Cross-System Health Deferral:** Autonomous observation task postponement when spacecraft subsystem health risk $r_h \ge 0.50$.
5. **Standard Geospatial Export:** Generation of RGBA transparent mask overlays and RFC 7946 GeoJSON boundary footprints for web GIS integration.


In [1]:
import os
import sys

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.insert(0, os.path.abspath('..'))

from data.synthetic.disaster_gen import generate_synthetic_chip_arrays
from ml.disaster.change_detector import SARChangeDetector
from ml.disaster.geo_export import export_geojson_footprint, export_overlay_png
from ml.disaster.module import DisasterRiskModule
from ml.disaster.unet import UNetSegmenter, compute_dice_f1, compute_iou

print('Environment and Disaster AI modules initialized successfully.')

## 1. Synthetic Sentinel-1 SAR Chip Generation & U-Net Segmentation
We generate a synthetic dual-polarization/backscatter SAR chip representing water inundation across varied terrain, and run U-Net flood segmentation.

In [2]:
raw_chip, true_mask = generate_synthetic_chip_arrays(size=128, water_ratio=0.30, seed=42)

model = UNetSegmenter(in_channels=1, base_filters=16)
# In production, model weights are loaded from disk or pre-fitted
pred_mask, pred_prob = model.predict_mask(raw_chip, threshold=0.5)
area_km2 = model.compute_surface_area_km2(pred_mask, resolution_m=10.0)

iou = compute_iou(pred_mask, true_mask)
dice = compute_dice_f1(pred_mask, true_mask)
print(f'Evaluated Chip Metrics -> IoU: {iou:.4f} | Dice (F1): {dice:.4f} | Detected Inundated Area: {area_km2:.2f} km^2')

## 2. Multi-Temporal SAR Change Detection
Compare pre-flood baseline SAR backscatter with post-flood acquisition to identify newly flooded terrain.

In [3]:
pre_chip, _ = generate_synthetic_chip_arrays(size=128, water_ratio=0.10, seed=10)
post_chip, _ = generate_synthetic_chip_arrays(size=128, water_ratio=0.35, seed=11)

detector = SARChangeDetector(threshold_db_drop=0.25, resolution_m=10.0)
change_result = detector.detect_flood_extent(pre_chip, post_chip)

print('Change Detection Results:')
print(f' - Newly flooded area: {change_result.newly_flooded_km2:.2f} km^2')
print(f' - Expansion ratio: {change_result.expansion_ratio:.2f}x')
print(f' - Total post-event water area: {change_result.total_water_km2:.2f} km^2')

## 3. Dynamic Observation Priority Scoring & Health Safety Cross-Effects
Evaluate observation priority $r_e$ and test health risk override ($r_h \ge 0.50 \implies$ deferral).

In [4]:
module = DisasterRiskModule()
module.fit()

features = {
    'area_km2': 180.0,
    'hours_ago': 1.5,
    'cloud_pct': 5.0,
    'population_proxy': 0.8,
}

nom_score = module.score(features, spacecraft_health_score=0.10)
print('Nominal Health (r_h = 0.10):')
print(f'  Observation Priority: {nom_score.score:.3f}')
print(f'  Recommendation: {nom_score.evidence["recommendation"]}')

deg_score = module.score(features, spacecraft_health_score=0.65)
print('\nDegraded Health (r_h = 0.65):')
print(f'  Observation Priority: {deg_score.score:.3f}')
print(f'  Recommendation: {deg_score.evidence["recommendation"]}')
print(f'  Deferral Flag: {deg_score.evidence["deferral_flag"]}')

## 4. Geospatial Export: RGBA Overlay PNG & GeoJSON Footprint

In [5]:
os.makedirs('data/synthetic/masks', exist_ok=True)
os.makedirs('data/synthetic/tiles', exist_ok=True)

png_path = 'data/synthetic/masks/sample_flood_overlay.png'
export_overlay_png(pred_mask, png_path)
print(f'Exported RGBA PNG overlay -> {png_path} (exists: {os.path.exists(png_path)})')

geojson_path = 'data/synthetic/tiles/sample_flood_footprint.geojson'
footprint = export_geojson_footprint(
    pred_mask,
    bounds=(20.5, 85.0, 20.9, 85.4),
    output_path=geojson_path,
    properties={'event': 'Cyclone Flooding', 'sensor': 'Sentinel-1 SAR'},
)
print(f'Exported GeoJSON footprint -> {geojson_path}')
print(f'GeoJSON type: {footprint["type"]}, features: {len(footprint["features"])}')